# 01 — `nn.Module` Basics: Linear, Embedding, LayerNorm, Dropout, and a Transformer Block

**Overview:** rebuild the manual-tensor 2-layer MLP from `19_pytorch_autograd` as a proper
`nn.Module`; demonstrate each core layer type individually; rebuild `17_deep_learning_architectures`'s
Transformer block using `nn.MultiheadAttention`.

**Required libraries:** `torch`.

Connects to: `19_pytorch_autograd` (same math, now organized) and `17_deep_learning_architectures`
(the Transformer block, now using PyTorch's built-in attention). Leads to:
`21_pytorch_data`.

### Abbreviations used in this notebook

| Term | Full form | What it is |
|---|---|---|
| **MLP** | Multi-Layer Perceptron | Feed-forward network of fully connected layers with non-linear activations. |
| **ReLU** | Rectified Linear Unit | Activation max(0, x). |
| **ID** | Identifier | Integer index of a token (token ID) or record. |
| **FFN** | Feed-Forward Network | Per-token two-layer MLP inside a Transformer block. |

## 0. Environment

In [1]:
import torch
import torch.nn as nn
print("torch:", torch.__version__)
torch.manual_seed(42)


torch: 2.14.1


## 1. `nn.Module` basics: the 2-layer MLP, properly organized

🎯 **Interview Tip:** `super().__init__()` MUST be called first -- it's what lets `nn.Module`
register everything assigned to `self` in `__init__` as a tracked sub-module/parameter.

In [2]:
class TwoLayerMLP(nn.Module):
    def __init__(self, in_features, hidden_units, out_classes):
        super().__init__()   # required -- registers this module with nn.Module's bookkeeping
        self.layer1 = nn.Linear(in_features, hidden_units)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(hidden_units, out_classes)

    def forward(self, x):
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        return x

model = TwoLayerMLP(in_features=5, hidden_units=8, out_classes=3)
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"\ntotal parameters: {total_params}")
# layer1: 5*8 + 8 = 48, layer2: 8*3 + 3 = 27 -> 75 total
assert total_params == (5 * 8 + 8) + (8 * 3 + 3)


TwoLayerMLP(
  (layer1): Linear(in_features=5, out_features=8, bias=True)
  (activation): ReLU()
  (layer2): Linear(in_features=8, out_features=3, bias=True)
)

total parameters: 75


## 2. Verifying it matches the manual-tensor version from `19_pytorch_autograd`

💡 **Key Idea:** `nn.Module` doesn't change the math -- loading the SAME weights into both
versions must produce identical output.

In [3]:
X = torch.randn(4, 5)

# Manually set this nn.Module's weights to specific values.
with torch.no_grad():
    model.layer1.weight.copy_(torch.randn(8, 5) * 0.3)
    model.layer1.bias.zero_()
    model.layer2.weight.copy_(torch.randn(3, 8) * 0.3)
    model.layer2.bias.zero_()

module_output = model(X)

# Manual-tensor version, using the SAME weights (nn.Linear stores weight as (out, in), so .T to match X @ W).
manual_output = torch.relu(X @ model.layer1.weight.T + model.layer1.bias) @ model.layer2.weight.T + model.layer2.bias

print("nn.Module output:\n", module_output)
print("\nmanual-tensor output:\n", manual_output)
assert torch.allclose(module_output, manual_output, atol=1e-6)
print("\nidentical, as expected -- nn.Module is purely organizational.")


nn.Module output:
 tensor([[-0.3651,  0.0046,  0.4287],
        [ 0.3076,  0.0628,  0.3259],
        [-0.0083, -0.0518,  0.1402],
        [ 0.2490,  0.2686,  0.2648]], grad_fn=<AddmmBackward0>)

manual-tensor output:
 tensor([[-0.3651,  0.0046,  0.4287],
        [ 0.3076,  0.0628,  0.3259],
        [-0.0083, -0.0518,  0.1402],
        [ 0.2490,  0.2686,  0.2648]], grad_fn=<AddBackward0>)

identical, as expected -- nn.Module is purely organizational.


## 3. `nn.Embedding`: a lookup table for token IDs

In [4]:
vocab_size, embedding_dim = 100, 16
embedding_layer = nn.Embedding(vocab_size, embedding_dim)

token_ids = torch.tensor([[5, 20, 3, 0], [7, 7, 99, 1]])   # (batch=2, seq_len=4)
embedded = embedding_layer(token_ids)
print("token_ids shape:", token_ids.shape)
print("embedded shape: ", embedded.shape, "== (batch, seq_len, embedding_dim)")

assert embedded.shape == (2, 4, embedding_dim)
# Same token ID should map to the same embedding vector, wherever it appears.
assert torch.allclose(embedded[1, 0], embedded[1, 1])   # both are token 7


token_ids shape: torch.Size([2, 4])
embedded shape:  torch.Size([2, 4, 16]) == (batch, seq_len, embedding_dim)


## 4. `nn.LayerNorm` matches the from-scratch version from `17_deep_learning_architectures`

In [5]:
def layer_norm_manual(x, eps=1e-5):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    return (x - mean) / torch.sqrt(var + eps)

x = torch.randn(4, 16)
torch_layer_norm = nn.LayerNorm(16, elementwise_affine=False)  # disable learned scale/shift for a fair comparison

torch_result = torch_layer_norm(x)
manual_result = layer_norm_manual(x)
print("max abs difference:", (torch_result - manual_result).abs().max().item())
assert torch.allclose(torch_result, manual_result, atol=1e-5)


max abs difference: 2.384185791015625e-07


## 5. `nn.Dropout`: different behavior in train vs eval mode

⚠️ **Common Mistake:** forgetting `.eval()` before inference, leaving dropout randomly zeroing
activations at serving time.

In [6]:
dropout = nn.Dropout(p=0.5)
x = torch.ones(10)

dropout.train()   # training mode: dropout IS active
train_outputs = [dropout(x) for _ in range(5)]
print("train mode outputs (randomly zeroed):\n", torch.stack(train_outputs))

dropout.eval()    # eval mode: dropout is a no-op
eval_outputs = [dropout(x) for _ in range(5)]
print("\neval mode outputs (always identical to input):\n", torch.stack(eval_outputs))

assert not all(torch.equal(train_outputs[0], t) for t in train_outputs[1:])   # train: varies
assert all(torch.equal(eval_outputs[0], t) for t in eval_outputs)              # eval: deterministic
assert torch.equal(eval_outputs[0], x)                                         # eval: unchanged


train mode outputs (randomly zeroed):
 tensor([[2., 0., 0., 2., 2., 0., 0., 0., 0., 2.],
        [0., 2., 0., 0., 2., 0., 2., 2., 2., 2.],
        [0., 2., 0., 2., 0., 2., 0., 0., 2., 2.],
        [0., 0., 2., 2., 2., 0., 2., 0., 0., 0.],
        [0., 2., 2., 2., 0., 0., 2., 2., 0., 2.]])

eval mode outputs (always identical to input):
 tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])


## 6. A Transformer block using `nn.MultiheadAttention` + `nn.LayerNorm` + `nn.Linear`

💡 **Key Idea:** same structure as `17_deep_learning_architectures`'s from-scratch version, now
using PyTorch's production implementations.

In [7]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, attn_mask=None):
        attn_out, _ = self.attention(x, x, x, attn_mask=attn_mask)   # self-attention: Q=K=V=x
        x = self.norm1(x + attn_out)      # residual + norm
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)        # residual + norm
        return x

d_model, num_heads, d_ff, seq_len = 16, 4, 32, 5
block = TransformerBlock(d_model, num_heads, d_ff)

token_embeddings = torch.randn(2, seq_len, d_model)   # (batch=2, seq_len=5, d_model=16)
output = block(token_embeddings)
print("input shape: ", token_embeddings.shape)
print("output shape:", output.shape)
assert output.shape == token_embeddings.shape   # shape-preserving, so blocks stack

# Causal masking, exactly as in 17_deep_learning_architectures.
causal_mask = torch.triu(torch.full((seq_len, seq_len), float("-inf")), diagonal=1)
causal_output = block(token_embeddings, attn_mask=causal_mask)
print("\nwith a causal mask, output shape is still:", causal_output.shape)
assert causal_output.shape == token_embeddings.shape


input shape:  torch.Size([2, 5, 16])
output shape: torch.Size([2, 5, 16])

with a causal mask, output shape is still: torch.Size([2, 5, 16])


## 7. Connection to what's next

`21_pytorch_data` builds the `Dataset`/`DataLoader` that will feed batches into exactly this
kind of `nn.Module`, and `22_pytorch_training` wires it all into a full training loop with an
optimizer updating `model.parameters()`.

## 8. Interview questions
1. Why does `nn.Module` require calling `super().__init__()` before registering sub-layers?
2. What does `model.parameters()` return, and why does an optimizer need it?
3. Why does `nn.Dropout` behave differently in `.train()` vs `.eval()` mode?
4. What does `nn.Embedding` return for a batch of token ID sequences, shape-wise?
5. Why does wrapping an attention + FFN block in an `nn.Module` class make it easy to stack multiple blocks?

## 9. Exercises
1. Add a second `TransformerBlock` and stack two of them, verifying the final output shape is unchanged.
2. Build an `nn.Sequential` version of the 2-layer MLP and verify it produces the same output as the `nn.Module` subclass version, given the same weights.
3. Print `model.named_parameters()` for the `TransformerBlock` and identify which parameters belong to the attention layer vs the FFN.

## 10. Summary

✅ `nn.Module` organizes layers and automatically registers their parameters;
`nn.Linear`/`nn.Embedding`/`nn.LayerNorm`/`nn.Dropout` are the production building blocks for
everything built from scratch in Day 3 so far. Next: `21_pytorch_data`.